# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/widadfatimakhan/flyrank-internship-ml/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Which pages should an editor review first?** A position-relative CTR queue, ranked by a
forward-validated model and explained by a frozen rule.

**Deployed paper:** https://widadfatimakhan.github.io/flyrank-internship-ml/

**Repository:** https://github.com/widadfatimakhan/flyrank-internship-ml

This notebook is an **assembly** notebook, not a compute notebook. Seven weeks of analysis already
ran against the 78.8M-row warehouse and left committed receipts behind. This notebook reads those
receipts, rebuilds the tables and figures the paper embeds, and checks that every number on the
deployed page traces to a file in the repository.

That is deliberate. If the paper's numbers can only be reproduced by re-running seven notebooks
against a gated dataset, they are not really checkable. Reading them back from committed receipts
is what makes the result-to-notebook map in the paper mean something.

| Paper section | Here |
|---|---|
| Abstract, Introduction | §1 Question |
| Data | §2 |
| Methodology | §3 |
| Results | §4 |
| Limitations | §5 |
| Ranked recommendations | §6 |
| Figures and tables the page embeds | §7 |
| — | §8 ML-12: demo outline, social cut, employer summary |

In [1]:
# --- Load the committed receipts ---------------------------------------------
# Works from a clone (local paths) or from a fresh Colab session (clones the public repo).
import json, os, subprocess, sys
import pandas as pd

REPO_URL = "https://github.com/widadfatimakhan/flyrank-internship-ml"
CANDIDATES = ["work/outputs", "../outputs", "flyrank-internship-ml/work/outputs"]

def find_outputs():
    for p in CANDIDATES:
        if os.path.isdir(p) and any(f.endswith(".json") for f in os.listdir(p)):
            return p
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], capture_output=True)
    p = "flyrank-internship-ml/work/outputs"
    return p if os.path.isdir(p) else None

OUT = find_outputs()
assert OUT, "receipts not found -- clone the repo or run from the repo root"
FIG = os.path.join(os.path.dirname(OUT), "figures")

R = {}
for k, f in [("ml04", "ml04_data_contract_receipts.json"),
             ("ml05", "ml05_feature_vector_receipts.json"),
             ("ml06", "ml06_signal_audit_receipts.json"),
             ("ml07", "ml07_baseline_receipts.json"),
             ("ml08", "ml08_model_receipts.json"),
             ("ml09", "ml09_validation_audit_receipts.json"),
             ("ml10", "ml10_playbook_receipts.json")]:
    with open(os.path.join(OUT, f)) as fh:
        R[k] = json.load(fh)

print(f"receipts directory: {OUT}")
print(f"loaded {len(R)} receipts: {', '.join(R)}")
print(f"figures directory : {FIG} "
      f"({'found' if os.path.isdir(FIG) else 'MISSING -- see section 7'})")

receipts directory: flyrank-internship-ml/work/outputs
loaded 7 receipts: ml04, ml05, ml06, ml07, ml08, ml09, ml10
figures directory : flyrank-internship-ml/work/figures (found)


## 1. Question

*The research question and the decision it supports.*

**The decision.** One FlyRank content editor, once a month, choosing roughly **fifty pages** to open
out of a portfolio of tens of thousands. With a budget that small, the *order* of the queue is the
entire product — which is why precision@50 was fixed as the metric in Week 4, before any model was
trained.

**The research question.** Of the pages already flagged as converting worse than their
position-matched peers, which will **still** be under-converting next month — and therefore deserve
an editor's hour, rather than fixing themselves?

**Why a fixed rule cannot answer it.** A page at position 2 earning 1% CTR is doing badly; a page at
position 40 earning 1% is doing well. What counts as good depends on rank, so the comparison has to
be computed from the data. Measured on this portfolio, a flat "flag anything under 1% CTR" rule
flags **96.7%** of pages — it returns the whole portfolio, unsorted.

**What this is.** Decision-support for what to open first. Not a diagnosis of what is wrong with a
page, and not evidence that editing one changes its outcome.

In [2]:
pop = R["ml08"]["population"]
print("THE DECISION FRAME")
print(f"  decision moment      : {R['ml08']['design']['decision_moment']}")
print(f"  features from        : {R['ml08']['design']['features_month']}")
print(f"  outcome from         : {R['ml08']['design']['outcome_month']}")
print(f"  review budget        : {R['ml10']['queue']['review_budget']} pages/month")
print(f"  metric               : precision@50, fixed in Week 4 before training")
print(f"\n  eligible pages       : {pop['march_eligible']:,}")
print(f"  with observed outcome: {pop['with_april_outcome']:,} across {pop['clients']} clients")
print(f"  base rate            : {pop['base_rate']:.3f}  "
      f"<- about 9 in 10 flagged problems resolve on their own")
print(f"\n  label: {R['ml08']['design']['label']}")
print(f"  label type: {R['ml08']['design']['label_type']}")

THE DECISION FRAME
  decision moment      : 2026-04-01
  features from        : 2026-03
  outcome from         : 2026-04
  review budget        : 50 pages/month
  metric               : precision@50, fixed in Week 4 before training

  eligible pages       : 61,881
  with observed outcome: 60,942 across 34 clients
  base rate            : 0.095  <- about 9 in 10 flagged problems resolve on their own

  label: would the ML-07 rule flag this page again in April (gap >= 0.1pp and missed clicks >= 10)
  label type: OBSERVED outcome (first in this project), not an authored proxy


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release.** `FlyRank/internship-warehouse` — a gated, pseudonymized derivative of production Search
Console and Analytics data. No URLs, page titles or raw queries exist in it.

**Tables used.** `fact_content_daily_performance` (78,835,655 rows, one row per page per day per
client) for all features and outcomes; `dim_content` (519,606 rows) for page attributes, joined only
after its key was tested unique and its coverage measured; `dim_clients` read once for history
coverage, never as a feature.

**Windows.** Features March 2026, outcomes April 2026, training frame February→March. June 2026 is
**sealed** — the panel's final month, reserved as the natural future outcome window.

**Eligibility, each rule with its reason.** ≥500 impressions (below 100, one extra click moves CTR by
more than the entire flagging threshold — a gap there is not a weak measurement, it is not a
measurement); ≥5 active days (volatility and momentum are meaningless otherwise); a valid position.

**Excluded, each with a why.** All 14 GA4-side columns — availability is three-valued and tracks the
client, and among instrumented clients 93.9% of page-days still carry no GA4 data. The query table —
its fixed 90-day window overlaps the outcome window. Identifiers — grouping keys only. Product flags
— an existing system's answers, and predicting an answer from its own inputs is circular. URLs and
titles — removed upstream for privacy, which is also why this work can say a page under-converts but
never why.

**The correction worth publishing.** 264,737 rows carried a position below 1, which is impossible.
Rather than filtering them as corrupt, the column's provenance was tested: it equals
`sum_position / impressions` on every row, and Search Console documents `sum_position` as
**zero-based**. A page reading 0.06 sits at true position 1.06 — and the original guard was silently
deleting 452 of the best-ranked pages in the dataset. A `+1` correction is applied everywhere.

In [3]:
d = R["ml04"]["verified"]
print("DATA CONTRACT, verified by query (ML-04)")
print(f"  grain violations found        : {d['Q1_grain_violations']}  (0 = one row per page-day-client)")
print(f"  page-day rows in the window   : {d['Q2_page_day_rows']:,}")
print(f"  distinct pages / clients      : {d['Q2_distinct_pages']:,} / {d['Q2_distinct_clients']}")
print(f"  date span                     : {d['Q2_date_span'][0]} .. {d['Q2_date_span'][1]}")
print(f"  position convention           : {d.get('position_convention', 'zero-based; +1 applied')}")
print(f"\nTHE TWO KINDS OF NOTHING (why availability flags need IS TRUE, not = TRUE)")
print(f"  rows a '= FALSE' filter would silently drop: {d['Q3_ga4_nulls_missed_by_equals_false']:,}")
print(f"\nPOPULATION FUNNEL")
print(f"  eligible                      : {pop['march_eligible']:,}")
print(f"  with a readable outcome       : {pop['with_april_outcome']:,}")
print(f"  dropped, disclosed            : {pop['dropped_no_april_outcome']:,} "
      f"({pop['dropped_no_april_outcome']/pop['march_eligible']:.1%}) -- selection using "
      f"outcome-window information, stated rather than hidden")

DATA CONTRACT, verified by query (ML-04)
  grain violations found        : 0  (0 = one row per page-day-client)
  page-day rows in the window   : 9,841,378
  distinct pages / clients      : 331,437 / 55
  date span                     : 2026-03-01 .. 2026-03-31
  position convention           : gsc_avg_position is ZERO-BASED (= sum/impressions); +1 applied

THE TWO KINDS OF NOTHING (why availability flags need IS TRUE, not = TRUE)
  rows a '= FALSE' filter would silently drop: 3,018,741

POPULATION FUNNEL
  eligible                      : 61,881
  with a readable outcome       : 60,942
  dropped, disclosed            : 939 (1.5%) -- selection using outcome-window information, stated rather than hidden


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**The label is an observed outcome, not an authored proxy.** Earlier weeks ranked by a rule the
author defined; a model trained on that would be learning arithmetic back. Here the label is
forward-looking: *would the frozen Week-4 rule flag this page again in April?* Same thresholds, one
month later, applied to what actually happened.

**And that change makes prior-window clicks legal.** Leakage is not a property of a column — it is a
relationship between a column and a label. Against the same-window proxy, `clicks` was fatal
(0.167 → 0.902). Against an April label, March clicks are ordinary history. The notebook tested this
rather than assuming it, by fitting one model with prior-window features and one without.

**Eleven features**, all computable from `report_date <= 2026-03-31`: traffic shape, within-client
scale, page attributes, and prior-window outcome. Position stays a *grouping* variable, not a
feature, because it defines the peer group whose baseline the target subtracts.

**Validation solves two separate problems.** Time is handled by construction — features end 03-31,
outcomes start 04-01. Groups are handled by the split: `GroupKFold(5)` on client, asserted disjoint,
because client pooled CTR varies ~10× and a random split would let the model recognise the client
instead of learning the pattern.

**Leakage as a method, not an apology.** Three boundaries checked in code, and two positive controls
proving the detector can fire at all.

In [4]:
print("FEATURES")
for i, f in enumerate(R["ml05"]["features"]["names"], 1):
    print(f"  {i:>2}. {f}")
print(f"\n  dim_content joined: {R['ml05']['features']['dim_content_joined']} "
      f"(key tested unique, coverage measured, before any join)")

lk = R["ml05"]["leakage_hunt"]
print(f"\nLEAKAGE HUNT (ML-05) -- correlation test with a KNOWN leak as positive control")
print(f"  control (clicks, a known leak) : {lk['control_clicks_31d_spearman']:+.3f}")
print(f"  strongest honest feature       : {lk['strongest_honest_feature_spearman']:+.3f} "
      f"({lk['strongest_honest_feature']})")
print(f"  -> the control lights up and no honest feature approaches it, so the ALL-CLEAR means "
      f"something")
print(f"  peer baseline reconstructed from position alone: "
      f"{lk['peer_pp_from_position_spearman']:.3f} -> position stays CONTEXT, not a feature")

la = R["ml09"]["leakage_audit"]
print(f"\nLEAKAGE AUDIT (ML-09) -- the timeline, checked in code")
for k in ["timeline_ok", "no_outcome_columns", "no_product_flags"]:
    print(f"  {'PASS' if la[k] else 'FAIL'}  {k}")
print(f"  SABOTAGE TEST: injecting an outcome-month column moved ROC AUC by "
      f"{la['sabotage_test_auc_gain']:+.4f} -> detector fires: {la['detector_fires']}")
print(f"  population selection DISCLOSED : {la['population_dropped_no_outcome']:,} pages dropped")

print(f"\nVALIDATION")
print(f"  {R['ml08']['design']['validation']}, seed {R['ml08']['design']['seed']}")
sg = R["ml08"]["split_gap"]
print(f"  random split AUC {sg['auc_random_split']:.3f} vs grouped {sg['auc_grouped_split']:.3f} "
      f"-> {sg['auc_random_split']-sg['auc_grouped_split']:+.3f} is the memorisation a careless "
      f"split would have sold as skill")
hs = R["ml09"]["honest_split"]
print(f"  time-forward: {hs['after']}")
print(f"  base rate moved {hs['base_rate_train']:.3f} -> {hs['base_rate_deploy']:.3f} between "
      f"fitting and use; {hs['deploy_clients_new']} of "
      f"{hs['deploy_clients_new']+hs['deploy_clients_seen_in_training']} evaluated clients were new")

FEATURES
   1. log_impressions_31d
   2. days_with_impressions_31d
   3. position_volatility_31d
   4. top_day_impression_share
   5. momentum_log14v14
   6. impressions_share_of_client
   7. log_word_count
   8. content_meta_missing
   9. cat_feedly article
  10. cat_keyword article

  dim_content joined: True (key tested unique, coverage measured, before any join)

LEAKAGE HUNT (ML-05) -- correlation test with a KNOWN leak as positive control
  control (clicks, a known leak) : -0.691
  strongest honest feature       : +0.181 (impressions_share_of_client)
  -> the control lights up and no honest feature approaches it, so the ALL-CLEAR means something
  peer baseline reconstructed from position alone: 0.859 -> position stays CONTEXT, not a feature

LEAKAGE AUDIT (ML-09) -- the timeline, checked in code
  PASS  timeline_ok
  PASS  no_outcome_columns
  PASS  no_product_flags
  SABOTAGE TEST: injecting an outcome-month column moved ROC AUC by +0.0539 -> detector fires: True
  population s

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Every row is scored on the **same pages, same folds, same metric, same K, same tie policy**, with the
base rate printed beside it. The Logistic Regression row is a baseline that was *rejected*, kept
visible on purpose — a paper that shows only the baselines it beat invites the question of what it
hid.

In [5]:
rows = [{"ranking method": r["design"].split(". ", 1)[-1],
         "precision@50": r["precision@50"], "precision@200": r["precision@200"],
         "ROC AUC": r["ROC AUC"]} for r in R["ml09"]["honest_split"]["table"]]
lr = [r for r in R["ml08"]["comparison"] if r["model"].startswith("LogReg")][0]
rows.insert(2, {"ranking method": "Logistic Regression, shape features (REJECTED baseline)",
                "precision@50": lr["precision@50"], "precision@200": lr["precision@200"],
                "ROC AUC": lr["ROC AUC"]})
results = pd.DataFrame(rows)
base = R["ml08"]["population"]["base_rate"]
results["lift@50"] = (results["precision@50"] / base).round(2)
print(f"BASE RATE = {base:.3f}\n")
display(results)

bs = R["ml09"]["bootstrap_time_forward_vs_rule_at_200"]
print(f"\nIs the margin real? time-forward model minus frozen rule at K=200:")
print(f"  {bs['mean']:+.3f}  (95% bootstrap interval {bs['ci95'][0]:+.3f} to {bs['ci95'][1]:+.3f}, "
      f"{bs['resamples']} resamples)")
print(f"  the interval EXCLUDES zero -> unlikely to be resampling noise")

print(f"\nWHAT THE MODEL LEANS ON (permutation importance, held-out clients)")
for f in R["ml08"]["top_features"]:
    print(f"  {f['feature']:26} drop in AUC {f['drop_in_auc']:.4f}")
print("  one feature towering over the rest is the leakage symptom -- investigated, not celebrated:")
print("  the label needs April missed clicks >= 10, and missed clicks = impressions x gap, so page")
print("  size is close to a prerequisite for a positive label. See section 5.")

BASE RATE = 0.095



,ranking method,precision@50,precision@200,ROC AUC,lift@50
0,random ranking (floor),0.04,0.060,0.500,0.42
1,"ML-07 rule, no learning",0.90,0.875,0.864,9.48
2,"Logistic Regression, shape features (REJECTED ...",0.48,0.385,0.878,5.06
3,"BEFORE - grouped CV, same month",0.96,0.975,0.921,10.12
4,"AFTER - trained Feb->Mar, deployed unchanged",0.96,0.980,0.925,10.12



Is the margin real? time-forward model minus frozen rule at K=200:
  +0.103  (95% bootstrap interval +0.055 to +0.155, 300 resamples)
  the interval EXCLUDES zero -> unlikely to be resampling noise

WHAT THE MODEL LEANS ON (permutation importance, held-out clients)
  log_impressions_31d        drop in AUC 0.1244
  mar_gap_pp                 drop in AUC 0.0390
  log_clicks_31d             drop in AUC 0.0142
  mar_ctr_pp                 drop in AUC 0.0135
  position_volatility_31d    drop in AUC 0.0096
  one feature towering over the rest is the leakage symptom -- investigated, not celebrated:
  the label needs April missed clicks >= 10, and missed clicks = impressions x gap, so page
  size is close to a prerequisite for a positive label. See section 5.


## 5. Limitations

*What this work cannot claim.*

Each limitation attaches to a specific claim rather than serving as a general disclaimer.

1. **One forward window.** Exactly one train-then-deploy transition was evaluated.
2. **The label is size-influenced by construction.** The positive rate runs from 0.4% among pages
   under 1,000 impressions to 47.1% above 20,000. Ranking by size alone reaches precision@50 of 0.42
   and by gap alone 0.50 — so neither explains 0.96, but the exam is easier than it looks.
3. **The label cannot separate "fixed" from "died."** Two of the three most confident errors were
   pages whose traffic fell by more than half; their gaps stopped clearing the threshold because
   demand left.
4. **Survivor selection, disclosed.** 939 pages (1.5%) with no readable outcome were excluded.
5. **The peer comparison fails at positions 1–2** — 0.118% pooled CTR against 0.394% at 3–5, with
   every client tested reproducing it internally.
6. **Clients differ ~10×**, and per-client model AUC ran 0.600 to 0.990.
7. **No calibration.** The column is `model_score`, not confidence.
8. **A reproducibility caveat found by re-running.** Missing `ORDER BY` made seeded results unstable;
   pinning it moved an earlier headline from 1.00 to 0.96. The lower, stable number is reported.
9. **No causal evidence, and none is available** without an experiment.

In [6]:
print("OPEN CHECKS, collected from every committed receipt\n")
for k in ["ml04", "ml06", "ml08", "ml09", "ml10"]:
    oc = R[k].get("open_checks") or R[k].get("open_questions")
    if oc:
        print(f"{k.upper()}")
        for c in oc:
            print(f"   - {c}")
        print()

print("SIGNAL AUDIT VERDICTS (ML-06) -- beliefs tested, including the ones that failed")
for k, v in R["ml06"]["verdicts"].items():
    print(f"  {v:<18} {k}")
cs = R["ml06"]["client_spread"]
print(f"\nclient pooled CTR spread: {cs['lowest_pooled_ctr_pp']:.3f}% .. "
      f"{cs['highest_pooled_ctr_pp']:.3f}% ({cs['spread_x']:.0f}x) across "
      f"{cs['clients_over_floor']} clients")
tb = R["ml06"]["top_band_decomposition"]
print(f"positions 1-2 pooled CTR {tb['band_1_2_pooled_ctr_pp']:.3f}% vs positions 3-5 "
      f"{tb['band_3_5_pooled_ctr_pp']:.3f}%; {tb['clients_worse_at_1_2']} of "
      f"{tb['clients_in_both_bands_over_floor']} clients reproduce it against themselves")

OPEN CHECKS, collected from every committed receipt

ML04
   - CTR level is ~0.3% across ALL tiers vs ~2.78% documented, and top_3 sits below page_1 -- systemic, not an off-by-one
   - ga4_data_available FALSE conflates pre-start rows with zero-traffic rows
   - partial-history clients inside a fixed calendar window

ML09
   - only one deployment window - a full backtest needs many decision moments
   - no calibration check
   - no causal evidence: nothing was intervened on
   - June 2026 remains sealed

ML10
   - only one time-forward window evaluated
   - no calibration -- model_score is not a probability
   - no causal evidence: no page was edited as an experiment
   - June 2026 remains sealed

SIGNAL AUDIT VERDICTS (ML-06) -- beliefs tested, including the ones that failed
  FALSE              signal_1_normal_ctr_same_across_clients
  MIXED              signal_2_spiky_traffic_distorts_ctr
  CONFIRMED          signal_3_falling_impressions_worse_ctr
  CONFIRMED          flag_linked_to

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

**The frozen rule supplies the reason. The model supplies the order.** The rule is arithmetic a
person can check — *"CTR 0.09% against peers at 0.34%, a gap of 0.25pp worth about 45 clicks a
month."* The model is a probability nobody can interrogate, but it sorts better at the head of the
queue.

Four actions, each with a guardrail and a way to measure whether it worked:

| Action | Fires when | Guardrail |
|---|---|---|
| `SNIPPET_REVIEW` | flagged, steady traffic, not top-2 | check season, intent, and live client work **first** |
| `VERIFY_THEN_REVIEW` | flagged but spiky or a >20× swing | a broken tag looks exactly like a broken snippet |
| `MONITOR_ONLY` | impressions rising durably | a page gaining traffic is not a page to rewrite |
| `INVESTIGATE_QUIET_RISK` | high score, no rule reason — or position 1–2 | **a high score with no reason is a question, not an instruction** |

**The cooldown that was recorded as impossible.** An earlier receipt stated no optimisation history
existed in the release. It does. Implementing it stopped **4,383 rewrite instructions on pages
somebody had just rewritten** — and the hole is disclosed too: 48.2% of pages have no optimisation
date and pass the gate by default.

**Never automated:** no automatic rewriting, no automatic client-facing reporting, no treating
`model_score` as a probability.

In [7]:
q = R["ml10"]["queue"]
print("PROPOSED ACTION MIX across the eligible portfolio")
tot = sum(q["action_mix"].values())
for k, v in q["action_mix"].items():
    print(f"  {k:24} {v:>7,}  ({v/tot:5.1%})")
print(f"\n  actionable rows: {q['actionable_rows']:,} | review budget: {q['review_budget']}/month")

print("\nNO-GO POLICIES, with where each is enforced")
for p in R["ml10"]["no_go_policies"]:
    line = f"  [{p['status']:<24}] {p['policy']}"
    print(line)
    if "suppressed" in p:
        print(f"      -> {p['suppressed']:,} rewrite instructions suppressed; "
              f"{p['pages_with_no_optimisation_date']:,} pages have no date and pass by default")

perf = R["ml10"]["performance"]
print(f"\nVALUE, as derived arithmetic with its assumptions visible")
print(f"  in a {q['review_budget']}-page batch: rule finds ~{perf['precision_at_50_rule']*50:.0f}, "
      f"model ~{perf['precision_at_50_model']*50:.0f}, random ~{perf['base_rate']*50:.1f}")
print(f"  the model's contribution is the last {perf['extra_persistent_pages_per_50']:.0f} pages")
print(f"  cost: {q['review_budget']} reviews x ~20 min = ~{q['review_budget']*20/60:.0f} "
      f"editor-hours/month")
print(f"  ASSUMPTION: a descriptive scenario from one evaluated window, not expected future impact")

mon = R["ml10"]["monitoring"]
print(f"\nMONITORING ({mon['status']})")
print(f"  pre-release : population {mon['pre_release']['population_change']:+.1%}, "
      f"max feature PSI {mon['pre_release']['max_feature_psi']:.2f} "
      f"({mon['pre_release']['features_above_0_25']} above 0.25) -> TRIGGERED")
print(f"  post-label  : base rate {mon['post_label']['base_rate_train']:.3f} -> "
      f"{mon['post_label']['base_rate_deploy']:.3f} ({mon['post_label']['shift']:+.3f}), "
      f"model ahead of rule: {mon['post_label']['model_ahead_of_rule']} -> clear")
print("  AND the pre-release alarm was a POLICY BUG, not a model problem: February has 28 days and")
print("  March has 31, so active_days cannot exceed 28 in training and reaches 31 in scoring.")
print("  Correction adopted before use: normalise by days-in-month, compare year-over-year.")

PROPOSED ACTION MIX across the eligible portfolio
  NO_ACTION                 43,608  (71.6%)
  MONITOR_ONLY              14,025  (23.0%)
  VERIFY_THEN_REVIEW         1,795  ( 2.9%)
  SNIPPET_REVIEW             1,452  ( 2.4%)
  INVESTIGATE_QUIET_RISK        62  ( 0.1%)

  actionable rows: 17,334 | review budget: 50/month

NO-GO POLICIES, with where each is enforced
  [UPSTREAM (ML-04 contract)] thin-history pages never queued
  [UPSTREAM (ML-04 contract)] below-floor pages never queued
  [ASSERTED                ] durable risers monitor-only
  [ASSERTED                ] >20x swings verify first
  [ASSERTED                ] 90-day cooldown on recently optimised pages
      -> 4,383 rewrite instructions suppressed; 29,367 pages have no date and pass by default
  [ASSERTED                ] position 1-2 never gets a straight rewrite

VALUE, as derived arithmetic with its assumptions visible
  in a 50-page batch: rule finds ~45, model ~48, random ~4.7
  the model's contribution is the last 

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Two figures and one table. The figures were generated in ML-10 and committed to `work/figures/`;
this section verifies they exist, prints the captions the page uses, and writes the results table so
the paper and the notebook cannot drift apart.

**Deployment layout.** GitHub Pages serves `docs/`, and image paths inside it must be relative, so
the two PNGs are copied to `docs/img/` and referenced as `img/precision_at_k.png`.

In [8]:
import shutil
os.makedirs("docs/img", exist_ok=True)
os.makedirs(os.path.join(OUT), exist_ok=True)

print("FIGURES the deployed page embeds")
for name in ["precision_at_k.png", "action_mix.png"]:
    src = os.path.join(FIG, name)
    if os.path.exists(src):
        shutil.copy(src, os.path.join("docs/img", name))
        print(f"  OK  {name}  ({os.path.getsize(src)/1024:.0f} KB) -> docs/img/{name}")
    else:
        print(f"  MISSING  {name} -- expected at {src}")

print("\nCAPTIONS (stored in the ML-10 receipt, reused verbatim on the page)")
for k, v in R["ml10"]["figure_captions"].items():
    print(f"\n{k}:\n  {v}")

results.to_csv(os.path.join(OUT, "capstone_results_table.csv"), index=False)
print(f"\nwrote {OUT}/capstone_results_table.csv -- the table rendered in the paper's Results section")
print("\nPublic-safety pass on everything the page will serve:")
print("  no client names, URLs, page titles or raw queries appear in any figure, table or caption")
print("  identifiers are pseudonymous hashes, used for grouping and validation only")
print("  the HF token is read from an environment variable / Colab secret and is in no committed file")

FIGURES the deployed page embeds
  OK  precision_at_k.png  (69 KB) -> docs/img/precision_at_k.png
  OK  action_mix.png  (51 KB) -> docs/img/action_mix.png

CAPTIONS (stored in the ML-10 receipt, reused verbatim on the page)

figure_1:
  Figure 1. Precision@K for the model order and the frozen Week-4 rule on 60,942 pages across 34 clients, features from March 2026 and observed outcomes from April 2026; the model was fitted on February->March and applied unchanged. Higher is better. The dotted line is the base rate (0.095) -- what a random queue returns. The shaded band is the 50-page review budget, which is where the comparison matters. WHAT THIS DOES NOT PROVE: it does not show that any feature is a search ranking factor, and it does not show that editing a flagged page causes its gap to close. Predictive, not causal.

figure_2:
  Figure 2. Action mix across 60,942 eligible pages at the 2026-04-01 decision moment. Reason codes come from the frozen Week-4 rule; the order within each act

## 8. ML-12 — demo outline, social cut, employer summary

Same evidence, three doors. **The evidence, the numbers, the limitations and the methods never
change** — only the order and the emphasis do.

---

### A. Five-minute demo outline

| min | beat | what is on screen | the sentence |
|---|---|---|---|
| 0:00–0:45 | **The constraint** | portfolio size vs a 50-page budget | "An editor can open about fifty pages a month out of tens of thousands. So the order of the queue *is* the product." |
| 0:45–1:30 | **Why a rule alone fails** | the flat-threshold result | "A flat 1% CTR rule flags 96.7% of the portfolio. What counts as good depends on where you rank." |
| 1:30–2:15 | **The honest label** | the timeline diagram | "Instead of scoring my own rule, I asked what April actually did: would this page still be flagged? Base rate 9.5% — nine in ten fix themselves." |
| 2:15–3:15 | **The result** | Figure 1, budget band highlighted | "0.96 against the frozen rule's 0.90 and a base rate of 0.095 — on held-out clients, and on a month the model was fitted before. Bootstrap interval excludes zero." |
| 3:15–4:15 | **The judgement** | the split-boundary and ORDER BY slides | "A random split flatters this. I report the grouped number. And re-running moved my own headline from 1.00 to 0.96 — row order was unpinned. I report the lower, stable one." |
| 4:15–5:00 | **The playbook** | Figure 2 + a queue line | "The rule gives the reason, the model gives the order. 4,383 rewrite instructions were suppressed because someone had just rewritten those pages. Predictive, not causal." |

**If asked one question, make it this one:** *"why did you split by client?"* — the answer is the 10×
client CTR spread, and it opens everything else.

---

### B. Social post cut (methodology, one finding, one chart, one link)

> **Which pages should an editor review first?**
>
> I ranked a 50-page monthly review queue on 60,942 pages across 34 client portfolios, drawn from a
> 79M-row production search warehouse.
>
> **The finding:** a model ordering the queue measured precision@50 of 0.96 against a hand-written
> rule at 0.90 — where a random queue returns 0.095. That is *at the top of the queue*, on clients
> the model had never seen, in a month it was fitted before.
>
> **The method bit worth stealing:** I split by client, not by row. Client click-through rates in
> this portfolio vary about 10×, so a random split lets a model recognise the client instead of
> learning the pattern. On a related configuration, the same test read 0.909 random and 0.893
> grouped. I report the grouped number, because that is the one that matches real use.
>
> **What changes on Monday:** the rule explains, the model orders. Every queue item carries a
> checkable sentence — "CTR 0.09% vs peers at 0.34%, worth ~45 clicks/month" — and a guardrail. A
> 90-day cooldown stopped 4,383 instructions to rewrite pages somebody had just rewritten.
>
> **What it does not prove:** no feature here is shown to be a search ranking factor, and nothing
> shows that rewriting a page causes recovery. Predictive, not causal.
>
> [Figure 1] · Full paper, notebooks and receipts: [link]

---

### C. Employer three-sentence summary

> I built a monthly content-review ranking system on a 79-million-row pseudonymized production search
> warehouse — 60,942 eligible pages across 34 client portfolios — where the deliverable was a
> 50-page queue, so precision at the head of the list was the metric, fixed before any model existed.
> Validated on held-out clients and again on a month the model was fitted before, it measured
> precision@50 of 0.96 against a frozen rule baseline at 0.90 and a 0.095 base rate, with a bootstrap
> interval on the margin that excludes zero. The work I would actually talk about is the auditing:
> I caught a zero-based position convention that was silently deleting my best-ranked pages, found
> that an unpinned row order had inflated my own headline from 0.96 to 1.00, and shipped the lower
> number — because a result you cannot reproduce is not a result.

In [9]:
print("ML-12 ARTIFACTS -- one artifact, three doors\n")
doors = {
    "Door 1 - reviewer":  "credibility: the split boundary, the leakage method, the uncertainty, "
                          "the result-to-notebook map",
    "Door 2 - employer":  "judgement and ownership: the worse number that won, and every failure "
                          "that became a design decision",
    "Door 3 - community": "what changes on Monday: actions, guardrails, and 'predictive, not causal' "
                          "written on the chart",
}
for k, v in doors.items():
    print(f"  {k:22} {v}")

print("\nWHAT NEVER CHANGES BETWEEN DOORS")
for x in ["the evidence", "the numbers", "the limitations", "the methods"]:
    print(f"  - {x}")

print(f"\nHEADLINE, in the four safe words:")
print(f"  OBSERVED on {pop['with_april_outcome']:,} pages across {pop['clients']} clients, a Random "
      f"Forest fitted before the evaluated month")
print(f"  MEASURED precision@50 of "
      f"{R['ml09']['honest_split']['table'][-1]['precision@50']:.2f} against a frozen rule at "
      f"{R['ml09']['honest_split']['table'][1]['precision@50']:.2f} and a base rate of {base:.3f};")
print(f"  the improvement is DIRECTIONAL, not uniform (per-client AUC 0.600 to 0.990, one window);")
print(f"  the queue is DECISION-SUPPORT for what to open first -- no causal claim.")

ML-12 ARTIFACTS -- one artifact, three doors

  Door 1 - reviewer      credibility: the split boundary, the leakage method, the uncertainty, the result-to-notebook map
  Door 2 - employer      judgement and ownership: the worse number that won, and every failure that became a design decision
  Door 3 - community     what changes on Monday: actions, guardrails, and 'predictive, not causal' written on the chart

WHAT NEVER CHANGES BETWEEN DOORS
  - the evidence
  - the numbers
  - the limitations
  - the methods

HEADLINE, in the four safe words:
  OBSERVED on 60,942 pages across 34 clients, a Random Forest fitted before the evaluated month
  MEASURED precision@50 of 0.96 against a frozen rule at 0.90 and a base rate of 0.095;
  the improvement is DIRECTIONAL, not uniform (per-client AUC 0.600 to 0.990, one window);
  the queue is DECISION-SUPPORT for what to open first -- no causal claim.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
